# Phase 3: Bitcoin Price Analytics Dashboard
## Interactive Dashboard using Dash with Multiple Pages
Combining insights from Phase 1 (EDA) and Phase 2 (Visualizations)

## 1. Setup & Import Libraries

In [51]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os
from datetime import datetime, timedelta

import dash
from dash import dcc, html, Input, Output, State, callback
import dash_bootstrap_components as dbc


## 2. Load & Prepare Data

In [52]:
# Load data from bitcoin-prices-dataset
path = "bitcoin-prices-dataset"
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))

# Data preprocessing
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
print(f"\nColumns: {df.columns.tolist()}")

Dataset loaded successfully!
Shape: (5883, 7)
Date Range: 2010-01-01 to 2026-02-08

Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'PriceCategory']


## 3. Feature Engineering (Phase 2)

In [53]:
# Calculate key metrics that will drive our narrative

# Measures the percentage range of price movement within a day
df['Daily_Volatility'] = ((df['High'] - df['Low']) / df['Low']) * 100

# Direction and magnitude of daily movement
df['Price_Change'] = ((df['Close'] - df['Open']) / df['Open']) * 100

# Moving averages
df['MA_30'] = df['Close'].rolling(window=30).mean()
df['MA_50'] = df['Close'].rolling(window=50).mean()

# 30-day rolling average of volatility
df['Volatility_MA_30'] = df['Daily_Volatility'].rolling(window=30).mean()

# Year and Month for grouping analysis
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)

print(f"✅ Feature engineering completed!")
print(f"\nFeature Summary:")
print(df[['Daily_Volatility', 'Price_Change', 'Volatility_MA_30']].describe())

✅ Feature engineering completed!

Feature Summary:
       Daily_Volatility  Price_Change  Volatility_MA_30
count       5883.000000   5883.000000       5854.000000
mean           3.833889      1.553441          3.842383
std           77.290527     76.124849         14.059741
min            0.209324    -73.561781          1.616316
25%            1.522888      0.000000          2.022211
50%            2.092616      0.000000          2.121155
75%            2.696399      0.000000          2.237626
max         5544.002504   5472.875092        187.106743


## 4. Calculate Key Metrics

In [54]:
# Calculate key metrics for KPI cards
current_price = df['Close'].iloc[-1]
starting_price = df['Close'].iloc[0]
total_return = ((current_price - starting_price) / starting_price) * 100
avg_volatility = df['Daily_Volatility'].mean()

print(f"Current Price: ${current_price:,.2f}")
print(f"Starting Price: ${starting_price:,.4f}")
print(f"Total Return: {total_return:,.1f}%")
print(f"Average Volatility: {avg_volatility:.2f}%")
print(f"Data Points: {len(df):,}")

Current Price: $69,000.00
Starting Price: $0.3000
Total Return: 22,999,900.0%
Average Volatility: 3.83%
Data Points: 5,883


## 5. Initialize Dash App

In [55]:
# Initialize Dash app
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Color scheme for price categories
category_colors = {
    'Low': '#1f77b4',
    'Medium': '#ff7f0e',
    'High': '#d62728',
    'Very High': '#9467bd'
}

print("DONE: Dash app initialized successfully!")

DONE: Dash app initialized successfully!


## 6. Define App Layout

In [56]:
app.layout = dbc.Container([
    dcc.Location(id='url', refresh=False),
    
    # Header
    dbc.Row([
        dbc.Col([
            html.Div([
                html.H1("Bitcoin Price Analytics Dashboard", className="mb-0"),
                html.P("Interactive Analysis of Bitcoin Price Evolution (2010-Present)", 
                       className="text-muted")
            ], className="p-4")
        ])
    ], className="bg-light border-bottom mb-4"),
    
    # Navigation Tabs
    dbc.Row([
        dbc.Col([
            dcc.Tabs(id="tabs", value="tab-1", children=[
                dcc.Tab(label="📈 Price Evolution", value="tab-1", children=[]),
                dcc.Tab(label="📊 Market Analysis", value="tab-2", children=[]),
                dcc.Tab(label="⚠️ Risk & Volatility", value="tab-3", children=[]),
            ])
        ])
    ], className="mb-4"),
    
    # Page Content
    html.Div(id="page-content", className="mb-4"),
    
], fluid=True, className="py-4")

print("✅ Layout defined successfully!")

✅ Layout defined successfully!


## 7. Page 1: Price Evolution

In [57]:
def create_price_evolution_page():
    return dbc.Container([
        # KPI Cards
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H6("Current Price", className="text-muted small"),
                        html.H3(f"${current_price:,.2f}", className="text-primary fw-bold"),
                        html.P(f"as of {df['Date'].iloc[-1].strftime('%Y-%m-%d')}", 
                               className="text-muted small mb-0")
                    ])
                ], className="shadow-sm")
            ], md=3),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H6("Total Return", className="text-muted small"),
                        html.H3(f"{total_return:,.1f}%", 
                               className=f"fw-bold {'text-success' if total_return > 0 else 'text-danger'}"),
                        html.P(f"from ${starting_price:.4f}", className="text-muted small mb-0")
                    ])
                ], className="shadow-sm")
            ], md=3),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H6("Avg Volatility", className="text-muted small"),
                        html.H3(f"{avg_volatility:.2f}%", className="text-warning fw-bold"),
                        html.P("30-day rolling average", className="text-muted small mb-0")
                    ])
                ], className="shadow-sm")
            ], md=3),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H6("Data Points", className="text-muted small"),
                        html.H3(f"{len(df):,}", className="text-info fw-bold"),
                        html.P(f"{(df['Date'].max() - df['Date'].min()).days} trading days", 
                               className="text-muted small mb-0")
                    ])
                ], className="shadow-sm")
            ], md=3),
        ], className="mb-4"),
        
        # Controls
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.Label("Select Date Range:", className="fw-bold"),
                        dcc.DatePickerRange(
                            id='date-range-picker',
                            start_date=df['Date'].min(),
                            end_date=df['Date'].max(),
                            display_format='YYYY-MM-DD',
                            className="w-100"
                        )
                    ])
                ], className="shadow-sm")
            ], md=6),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.Label("Filter by Price Phase:", className="fw-bold"),
                        dcc.Dropdown(
                            id='phase-filter',
                            options=[{'label': 'All Phases', 'value': 'all'}] + 
                                   [{'label': phase, 'value': phase} 
                                    for phase in df['PriceCategory'].unique()],
                            value='all',
                            clearable=False
                        )
                    ])
                ], className="shadow-sm")
            ], md=6),
        ], className="mb-4"),
        
        # Main Price Chart
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        dcc.Loading(
                            id="loading-1",
                            type="default",
                            children=dcc.Graph(id='price-evolution-chart', 
                                             style={'height': '500px'})
                        )
                    ])
                ], className="shadow-sm")
            ])
        ], className="mb-4"),
        
        # Additional Metrics
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H5("Price Statistics", className="fw-bold mb-3"),
                        dcc.Graph(id='price-stats-chart', style={'height': '400px'})
                    ])
                ], className="shadow-sm")
            ], md=6),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H5("Daily Returns Distribution", className="fw-bold mb-3"),
                        dcc.Graph(id='returns-distribution-chart', style={'height': '400px'})
                    ])
                ], className="shadow-sm")
            ], md=6),
        ], className="mb-4"),
        
    ], fluid=True)

print("✅ Page 1 function created!")

✅ Page 1 function created!


## 8. Page 2: Market Analysis

In [58]:
def create_market_analysis_page():
    return dbc.Container([
        # Controls
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.Label("Moving Average Period:", className="fw-bold"),
                        dcc.Slider(
                            id='ma-period-slider',
                            min=7,
                            max=120,
                            step=1,
                            value=30,
                            marks={7: '7', 30: '30', 60: '60', 120: '120'},
                            tooltip={"placement": "bottom", "always_visible": True}
                        )
                    ])
                ], className="shadow-sm")
            ], md=6),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.Label("Volume Aggregation:", className="fw-bold"),
                        dcc.Dropdown(
                            id='volume-agg-dropdown',
                            options=[
                                {'label': 'Daily', 'value': 'daily'},
                                {'label': 'Weekly', 'value': 'weekly'},
                                {'label': 'Monthly', 'value': 'monthly'}
                            ],
                            value='daily',
                            clearable=False
                        )
                    ])
                ], className="shadow-sm")
            ], md=6),
        ], className="mb-4"),
        
        # Volume and Price Chart
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H5("Volume & Price Momentum", className="fw-bold mb-3"),
                        dcc.Loading(
                            id="loading-2",
                            type="default",
                            children=dcc.Graph(id='volume-price-chart', 
                                             style={'height': '500px'})
                        )
                    ])
                ], className="shadow-sm")
            ])
        ], className="mb-4"),
        
        # Phase Analysis
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H5("Market Phase Comparison", className="fw-bold mb-3"),
                        dcc.Graph(id='phase-comparison-chart', style={'height': '450px'})
                    ])
                ], className="shadow-sm")
            ], md=6),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H5("Volatility Trends by Year", className="fw-bold mb-3"),
                        dcc.Graph(id='volatility-by-year-chart', style={'height': '450px'})
                    ])
                ], className="shadow-sm")
            ], md=6),
        ], className="mb-4"),
        
    ], fluid=True)

print("✅ Page 2 function created!")

✅ Page 2 function created!


## 9. Page 3: Risk & Volatility

In [59]:
def create_risk_volatility_page():
    return dbc.Container([
        # Volatility Threshold Control
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.Label("Volatility Threshold (%)", className="fw-bold"),
                        dcc.Slider(
                            id='volatility-threshold-slider',
                            min=0,
                            max=10,
                            step=0.5,
                            value=3,
                            marks={0: '0%', 5: '5%', 10: '10%'},
                            tooltip={"placement": "bottom", "always_visible": True}
                        )
                    ])
                ], className="shadow-sm")
            ], md=6),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.Label("Statistical Summary", className="fw-bold"),
                        html.Div(id='volatility-stats-text', className="mt-3")
                    ])
                ], className="shadow-sm")
            ], md=6),
        ], className="mb-4"),
        
        # Volatility Chart
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H5("Daily Volatility Over Time", className="fw-bold mb-3"),
                        dcc.Loading(
                            id="loading-3",
                            type="default",
                            children=dcc.Graph(id='volatility-timeseries-chart', 
                                             style={'height': '500px'})
                        )
                    ])
                ], className="shadow-sm")
            ])
        ], className="mb-4"),
        
        # Risk Analysis
        dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H5("Risk Profile by Market Phase", className="fw-bold mb-3"),
                        dcc.Graph(id='risk-profile-chart', style={'height': '450px'})
                    ])
                ], className="shadow-sm")
            ], md=6),
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        html.H5("Volatility Distribution", className="fw-bold mb-3"),
                        dcc.Graph(id='volatility-distribution-chart', style={'height': '450px'})
                    ])
                ], className="shadow-sm")
            ], md=6),
        ], className="mb-4"),
        
    ], fluid=True)

print("✅ Page 3 function created!")

✅ Page 3 function created!


## 10. Callback: Page Routing

In [60]:
@app.callback(
    Output('page-content', 'children'),
    Input('tabs', 'value')
)
def display_page(tab):
    if tab == 'tab-1':
        return create_price_evolution_page()
    elif tab == 'tab-2':
        return create_market_analysis_page()
    elif tab == 'tab-3':
        return create_risk_volatility_page()

print("✅ Page routing callback registered!")

✅ Page routing callback registered!


## 11. Callbacks: Page 1 - Price Evolution

In [61]:
@app.callback(
    Output('price-evolution-chart', 'figure'),
    Input('date-range-picker', 'start_date'),
    Input('date-range-picker', 'end_date'),
    Input('phase-filter', 'value')
)
def update_price_evolution_chart(start_date, end_date, phase_filter):
    # Filter data
    mask = (df['Date'] >= start_date) & (df['Date'] <= end_date)
    filtered_df = df[mask].copy()
    
    if phase_filter != 'all':
        filtered_df = filtered_df[filtered_df['PriceCategory'] == phase_filter]
    
    fig = go.Figure()
    
    # Plot by phase with colors
    if phase_filter == 'all':
        for category in df['PriceCategory'].unique():
            mask_cat = filtered_df['PriceCategory'] == category
            fig.add_trace(go.Scatter(
                x=filtered_df[mask_cat]['Date'],
                y=filtered_df[mask_cat]['Close'],
                name=f'{category}',
                mode='lines',
                line=dict(color=category_colors.get(category, '#000000'), width=2),
                hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:,.2f}<extra></extra>'
            ))
    else:
        fig.add_trace(go.Scatter(
            x=filtered_df['Date'],
            y=filtered_df['Close'],
            name='Price',
            mode='lines',
            line=dict(color=category_colors.get(phase_filter, '#000000'), width=2),
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Price: $%{y:,.2f}<extra></extra>'
        ))
    
    # Add moving averages
    fig.add_trace(go.Scatter(
        x=filtered_df['Date'],
        y=filtered_df['MA_30'],
        name='30-Day MA',
        mode='lines',
        line=dict(color='rgba(0,255,0,0.5)', width=2, dash='dash'),
        hovertemplate='30-Day MA: $%{y:,.2f}<extra></extra>'
    ))
    
    fig.update_layout(
        title='Bitcoin Price Evolution with Market Phases',
        xaxis_title='Date',
        yaxis_title='Price (USD)',
        template='plotly_white',
        hovermode='x unified',
        height=500,
        yaxis=dict(type='log')
    )
    
    return fig

print("✅ Price evolution chart callback registered!")

✅ Price evolution chart callback registered!


In [62]:
@app.callback(
    Output('price-stats-chart', 'figure'),
    Input('date-range-picker', 'start_date'),
    Input('date-range-picker', 'end_date')
)
def update_price_stats_chart(start_date, end_date):
    mask = (df['Date'] >= start_date) & (df['Date'] <= end_date)
    filtered_df = df[mask].copy()
    
    # Calculate stats by phase
    stats = filtered_df.groupby('PriceCategory')['Close'].agg(['min', 'max', 'mean']).reset_index()
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=stats['PriceCategory'],
        y=stats['min'],
        name='Min Price',
        marker_color='#1f77b4'
    ))
    
    fig.add_trace(go.Bar(
        x=stats['PriceCategory'],
        y=stats['max'],
        name='Max Price',
        marker_color='#d62728'
    ))
    
    fig.add_trace(go.Bar(
        x=stats['PriceCategory'],
        y=stats['mean'],
        name='Avg Price',
        marker_color='#2ca02c'
    ))
    
    fig.update_layout(
        title='Price Statistics by Market Phase',
        xaxis_title='Price Category',
        yaxis_title='Price (USD)',
        barmode='group',
        template='plotly_white',
        height=400
    )
    
    return fig

print("✅ Price stats chart callback registered!")

✅ Price stats chart callback registered!


In [63]:
@app.callback(
    Output('returns-distribution-chart', 'figure'),
    Input('date-range-picker', 'start_date'),
    Input('date-range-picker', 'end_date')
)
def update_returns_distribution(start_date, end_date):
    mask = (df['Date'] >= start_date) & (df['Date'] <= end_date)
    filtered_df = df[mask].copy()
    
    fig = go.Figure()
    
    fig.add_trace(go.Histogram(
        x=filtered_df['Price_Change'],
        nbinsx=50,
        name='Daily Returns',
        marker_color='#1f77b4',
        hovertemplate='Return: %{x:.2f}%<br>Frequency: %{y}<extra></extra>'
    ))
    
    fig.update_layout(
        title='Distribution of Daily Price Changes',
        xaxis_title='Daily Return (%)',
        yaxis_title='Frequency',
        template='plotly_white',
        height=400
    )
    
    return fig

print("✅ Returns distribution chart callback registered!")

✅ Returns distribution chart callback registered!


## 12. Callbacks: Page 2 - Market Analysis

In [64]:
@app.callback(
    Output('volume-price-chart', 'figure'),
    Input('ma-period-slider', 'value'),
    Input('volume-agg-dropdown', 'value')
)
def update_volume_price_chart(ma_period, agg_type):
    plot_df = df.copy()
    
    # Calculate MA with selected period
    plot_df[f'MA_{ma_period}'] = plot_df['Close'].rolling(window=ma_period).mean()
    
    # Aggregate volume if needed
    if agg_type == 'weekly':
        plot_df['Week'] = plot_df['Date'].dt.isocalendar().week
        plot_df['YearWeek'] = plot_df['Date'].dt.strftime('%Y-W%W')
        volume_data = plot_df.groupby('YearWeek').agg({
            'Volume': 'sum',
            'Close': 'last',
            'Date': 'last',
            f'MA_{ma_period}': 'last'
        }).reset_index()
    elif agg_type == 'monthly':
        volume_data = plot_df.groupby('YearMonth').agg({
            'Volume': 'sum',
            'Close': 'last',
            'Date': 'last',
            f'MA_{ma_period}': 'last'
        }).reset_index()
        volume_data['Date'] = pd.to_datetime(volume_data['Date'])
    else:
        volume_data = plot_df.copy()
    
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        specs=[[{'secondary_y': False}], [{'secondary_y': False}]],
        row_heights=[0.6, 0.4],
        subplot_titles=('Bitcoin Price', f'Trading Volume ({agg_type.capitalize()})')
    )
    
    # Price with MA
    fig.add_trace(
        go.Scatter(
            x=volume_data['Date'],
            y=volume_data['Close'],
            name='Price',
            mode='lines',
            line=dict(color='#1f77b4', width=2),
            hovertemplate='Price: $%{y:,.2f}<extra></extra>'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=volume_data['Date'],
            y=volume_data[f'MA_{ma_period}'],
            name=f'{ma_period}-Day MA',
            mode='lines',
            line=dict(color='rgba(255,0,0,0.5)', width=2, dash='dash'),
            hovertemplate=f'{ma_period}-Day MA: $%{{y:,.2f}}<extra></extra>'
        ),
        row=1, col=1
    )
    
    # Volume bars
    colors = ['#2ca02c' if volume_data.iloc[i]['Close'] >= volume_data.iloc[i-1]['Close'] 
              else '#d62728' for i in range(len(volume_data))]
    
    fig.add_trace(
        go.Bar(
            x=volume_data['Date'],
            y=volume_data['Volume'],
            name='Volume',
            marker=dict(color=colors, opacity=0.6),
            hovertemplate='Volume: %{y:,.0f}<extra></extra>'
        ),
        row=2, col=1
    )
    
    fig.update_yaxes(title_text='Price (USD)', row=1, col=1, type='log')
    fig.update_yaxes(title_text='Volume (BTC)', row=2, col=1)
    fig.update_xaxes(title_text='Date', row=2, col=1)
    
    fig.update_layout(
        title_text=f'Volume & Price Momentum ({ma_period}-Day MA)',
        height=500,
        template='plotly_white',
        hovermode='x unified'
    )
    
    return fig

print("✅ Volume-price chart callback registered!")

✅ Volume-price chart callback registered!


In [65]:
@app.callback(
    Output('phase-comparison-chart', 'figure'),
    Input('tabs', 'value')
)
def update_phase_comparison_chart(tab):
    if tab != 'tab-2':
        return {}
    
    phase_summary = []
    for category in df['PriceCategory'].unique():
        mask = df['PriceCategory'] == category
        phase_summary.append({
            'Phase': category,
            'Days': mask.sum(),
            'Avg Volatility': df[mask]['Daily_Volatility'].mean(),
            'Avg Volume': df[mask]['Volume'].mean(),
            'Price Range': f"${df[mask]['Close'].min():.2f}-${df[mask]['Close'].max():.2f}"
        })
    
    phase_df = pd.DataFrame(phase_summary).sort_values('Days', ascending=True)
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Duration of Each Phase', 'Average Metrics'),
        specs=[[{'type': 'bar'}, {'type': 'bar'}]]
    )
    
    fig.add_trace(
        go.Bar(x=phase_df['Phase'], y=phase_df['Days'], name='Days',
               marker=dict(color='#1f77b4'), showlegend=False),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Bar(x=phase_df['Phase'], y=phase_df['Avg Volatility'], name='Volatility (%)',
               marker=dict(color='#d62728')),
        row=1, col=2
    )
    
    fig.update_yaxes(title_text='Days', row=1, col=1)
    fig.update_yaxes(title_text='Avg Volatility (%)', row=1, col=2)
    
    fig.update_layout(title_text='Market Phase Characteristics', height=450, template='plotly_white')
    
    return fig

print("✅ Phase comparison chart callback registered!")

✅ Phase comparison chart callback registered!


In [66]:
@app.callback(
    Output('volatility-by-year-chart', 'figure'),
    Input('tabs', 'value')
)
def update_volatility_by_year_chart(tab):
    if tab != 'tab-2':
        return {}
    
    yearly_vol = df.groupby('Year')['Daily_Volatility'].agg(['mean', 'std', 'max']).reset_index()
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=yearly_vol['Year'],
        y=yearly_vol['mean'],
        name='Avg Volatility',
        marker_color='#1f77b4',
        error_y=dict(type='data', array=yearly_vol['std']),
        hovertemplate='Year: %{x}<br>Avg: %{y:.2f}%<extra></extra>'
    ))
    
    fig.add_trace(go.Scatter(
        x=yearly_vol['Year'],
        y=yearly_vol['max'],
        name='Max Volatility',
        mode='lines+markers',
        line=dict(color='#d62728', width=2),
        marker=dict(size=8),
        hovertemplate='Year: %{x}<br>Max: %{y:.2f}%<extra></extra>'
    ))
    
    fig.update_layout(
        title='Yearly Volatility Trends',
        xaxis_title='Year',
        yaxis_title='Volatility (%)',
        template='plotly_white',
        height=450
    )
    
    return fig

print("✅ Volatility by year chart callback registered!")

✅ Volatility by year chart callback registered!


## 13. Callbacks: Page 3 - Risk & Volatility

In [67]:
@app.callback(
    Output('volatility-stats-text', 'children'),
    Input('volatility-threshold-slider', 'value')
)
def update_volatility_stats(threshold):
    high_vol_days = (df['Daily_Volatility'] > threshold).sum()
    total_days = len(df)
    percentage = (high_vol_days / total_days) * 100
    
    return [
        html.P(f"Days above {threshold}% volatility:", className="mb-2"),
        html.H4(f"{high_vol_days:,} days ({percentage:.1f}%)", className="text-danger fw-bold"),
        html.Hr(),
        html.P([
            html.Strong("Statistics: "),
            f"Min: {df['Daily_Volatility'].min():.2f}%, "
            f"Max: {df['Daily_Volatility'].max():.2f}%, "
            f"Mean: {df['Daily_Volatility'].mean():.2f}%"
        ], className="text-muted small")
    ]

print("✅ Volatility stats callback registered!")

✅ Volatility stats callback registered!


In [68]:
@app.callback(
    Output('volatility-timeseries-chart', 'figure'),
    Input('volatility-threshold-slider', 'value')
)
def update_volatility_timeseries(threshold):
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df['Date'],
        y=df['Daily_Volatility'],
        name='Daily Volatility',
        mode='lines',
        line=dict(color='#1f77b4', width=1),
        hovertemplate='Date: %{x|%Y-%m-%d}<br>Volatility: %{y:.2f}%<extra></extra>'
    ))
    
    fig.add_trace(go.Scatter(
        x=df['Date'],
        y=df['Volatility_MA_30'],
        name='30-Day MA',
        mode='lines',
        line=dict(color='#ff7f0e', width=2),
        hovertemplate='30-Day MA: %{y:.2f}%<extra></extra>'
    ))
    
    # Add threshold line
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Threshold: {threshold}%",
        annotation_position="right"
    )
    
    fig.update_layout(
        title='Bitcoin Daily Volatility Over Time',
        xaxis_title='Date',
        yaxis_title='Volatility (%)',
        template='plotly_white',
        hovermode='x unified',
        height=500
    )
    
    return fig

print("✅ Volatility timeseries callback registered!")

✅ Volatility timeseries callback registered!


In [69]:
@app.callback(
    Output('risk-profile-chart', 'figure'),
    Input('tabs', 'value')
)
def update_risk_profile_chart(tab):
    if tab != 'tab-3':
        return {}
    
    fig = go.Figure()
    
    for category in df['PriceCategory'].unique():
        mask = df['PriceCategory'] == category
        fig.add_trace(go.Box(
            y=df[mask]['Daily_Volatility'],
            name=category,
            marker_color=category_colors.get(category, '#000000'),
            hovertemplate='Volatility: %{y:.2f}%<extra></extra>'
        ))
    
    fig.update_layout(
        title='Risk Profile by Market Phase (Volatility Distribution)',
        yaxis_title='Daily Volatility (%)',
        template='plotly_white',
        height=450
    )
    
    return fig

print("✅ Risk profile chart callback registered!")

✅ Risk profile chart callback registered!


In [70]:
@app.callback(
    Output('volatility-distribution-chart', 'figure'),
    Input('tabs', 'value')
)
def update_volatility_distribution(tab):
    if tab != 'tab-3':
        return {}
    
    fig = go.Figure()
    
    fig.add_trace(go.Histogram(
        x=df['Daily_Volatility'],
        nbinsx=50,
        name='Daily Volatility',
        marker_color='#1f77b4',
        hovertemplate='Volatility: %{x:.2f}%<br>Frequency: %{y}<extra></extra>'
    ))
    
    fig.add_vline(
        x=df['Daily_Volatility'].mean(),
        line_dash="dash",
        line_color="green",
        annotation_text=f"Mean: {df['Daily_Volatility'].mean():.2f}%",
        annotation_position="top right"
    )
    
    fig.update_layout(
        title='Distribution of Daily Volatility',
        xaxis_title='Daily Volatility (%)',
        yaxis_title='Frequency',
        template='plotly_white',
        height=450
    )
    
    return fig

print("✅ Volatility distribution chart callback registered!")

✅ Volatility distribution chart callback registered!


## 14. Run the Dashboard

In [71]:
if __name__ == '__main__':
    print("\n" + "="*80)
    print("🚀 BITCOIN PRICE ANALYTICS DASHBOARD - STARTING")
    print("="*80)
    print(f"\n✅ Dashboard is now running at: http://localhost:8050/")
    print(f"\n📊 Features Available:")
    print(f"   • Page 1: Price Evolution (Date range, phase filters)")
    print(f"   • Page 2: Market Analysis (Volume trends, phase comparison)")
    print(f"   • Page 3: Risk & Volatility (Threshold, distributions)")
    print(f"\n💡 Data Summary:")
    print(f"   • Total Records: {len(df):,}")
    print(f"   • Date Range: {df['Date'].min().date()} to {df['Date'].max().date()}")
    print(f"   • Current Price: ${current_price:,.2f}")
    print(f"   • Total Return: {total_return:,.1f}%")
    print(f"\n⚠️  Press Ctrl+C to stop the server")
    print("="*80 + "\n")
    
    # Run the app
    app.run(debug=True, port=8050)


🚀 BITCOIN PRICE ANALYTICS DASHBOARD - STARTING

✅ Dashboard is now running at: http://localhost:8050/

📊 Features Available:
   • Page 1: Price Evolution (Date range, phase filters)
   • Page 2: Market Analysis (Volume trends, phase comparison)
   • Page 3: Risk & Volatility (Threshold, distributions)

💡 Data Summary:
   • Total Records: 5,883
   • Date Range: 2010-01-01 to 2026-02-08
   • Current Price: $69,000.00
   • Total Return: 22,999,900.0%

⚠️  Press Ctrl+C to stop the server




### Three Interactive Pages:
1. **Price Evolution**: Date ranges, phase filters, KPI cards
2. **Market Analysis**: Volume trends, moving averages, yearly volatility
3. **Risk & Volatility**: Threshold controls, distribution analysis


### To Run:
```python
# Install dependencies:
# pip install dash dash-bootstrap-components plotly pandas numpy

# Run this notebook and execute all cells
# Then run the last cell to start the server at http://localhost:8050/
```